In [9]:
# Cell 1: Setup Environment
# ===============================
#   SETUP — LLaVAProbe (Stable)
# ===============================
import os
import subprocess
import sys

# --- Upgrade pip first ---
print("📦 Upgrading pip...")
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])

# --- Install core dependencies with specific versions ---
print("\n📦 Installing dependencies...")
packages = [
    "transformers>=4.35.0",
    "accelerate>=0.24.0",
    "torch>=2.0.0",
    "timm",
    "einops",
    "scikit-learn",
    "matplotlib",
    "pillow",
    "seaborn",
    "xgboost",
    "hf_transfer",
    "sentencepiece",
    "protobuf"
]

for pkg in packages:
    print(f"Installing {pkg}...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg],
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        print(f"❌ Error installing {pkg}:")
        print(result.stderr)
    else:
        print(f"✅ {pkg} installed")

# --- Clone LLaVAProbe repo ---
REPO_URL = "https://github.com/itsloganmann/LLaVAProbe.git"
REPO_DIR = "LLaVAProbe"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL])
else:
    print("✅ Repository already exists, skipping clone")

# --- Move into repo ---
os.chdir(REPO_DIR)

# --- Create folder structure ---
os.makedirs("idea_4/experiments", exist_ok=True)
os.makedirs("idea_4/results", exist_ok=True)
os.makedirs("idea_4/docs", exist_ok=True)
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

print("✅ Setup complete — environment ready for Idea 4")




📦 Upgrading pip...
Defaulting to user installation because normal site-packages is not writeable



📦 Installing dependencies...
Installing transformers>=4.35.0...
✅ transformers>=4.35.0 installed
Installing accelerate>=0.24.0...
✅ accelerate>=0.24.0 installed
Installing torch>=2.0.0...
✅ torch>=2.0.0 installed
Installing timm...
✅ timm installed
Installing einops...
✅ einops installed
Installing scikit-learn...
✅ scikit-learn installed
Installing matplotlib...
✅ matplotlib installed
Installing pillow...
✅ pillow installed
Installing seaborn...
✅ seaborn installed
Installing xgboost...
✅ xgboost installed
Installing hf_transfer...
✅ hf_transfer installed
Installing sentencepiece...
✅ sentencepiece installed
Installing protobuf...
✅ protobuf installed


Cloning into 'LLaVAProbe'...


✅ Setup complete — environment ready for Idea 4


In [10]:
# Diagnostic cell - run this to check transformers version
import transformers
print(f"Transformers version: {transformers.__version__}")
print(f"Transformers location: {transformers.__file__}")

# List available imports
import pkgutil
transformers_imports = [name for _, name, _ in pkgutil.iter_modules(transformers.__path__)]
print(f"\nAutoProcessor available: {'AutoProcessor' in transformers_imports}")
print(f"LlavaForConditionalGeneration available: {'LlavaForConditionalGeneration' in dir(transformers)}")

# If version is old, we need to force upgrade
if transformers.__version__ < "4.35.0":
    print(f"\n❌ Transformers {transformers.__version__} is too old. Upgrading...")
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "--force-reinstall", "transformers>=4.35.0"])
    print("✅ Please restart kernel after upgrade")
else:
    print(f"\n✅ Transformers version is OK")

Transformers version: 4.57.3
Transformers location: /home/ubuntu/.local/lib/python3.10/site-packages/transformers/__init__.py

AutoProcessor available: False
LlavaForConditionalGeneration available: True

✅ Transformers version is OK


In [11]:
import transformers
from transformers import models

# Check what's available for LLaVA
print("Checking available LLaVA imports...")
print(f"Transformers version: {transformers.__version__}")

# Try to find LLaVA-related classes
import inspect
llava_items = [name for name in dir(transformers) if 'llava' in name.lower()]
print(f"\nLLaVA-related items in transformers: {llava_items}")

# Check models module
if hasattr(transformers, 'models'):
    if hasattr(transformers.models, 'llava'):
        print("\n✅ LLaVA module found!")
        llava_classes = [name for name in dir(transformers.models.llava) if not name.startswith('_')]
        print(f"Available classes: {llava_classes}")
    else:
        print("\n❌ No LLaVA module in transformers.models")

# Try direct import
try:
    from transformers.models.llava import LlavaForConditionalGeneration
    print("\n✅ Can import LlavaForConditionalGeneration from models.llava")
except ImportError as e:
    print(f"\n❌ Cannot import: {e}")

# Check for processor
try:
    from transformers import CLIPImageProcessor, LlamaTokenizerFast
    print("\n✅ Can import CLIPImageProcessor and LlamaTokenizerFast (components for LLaVA)")
except ImportError as e:
    print(f"\n❌ Cannot import components: {e}")

Checking available LLaVA imports...
Transformers version: 4.57.3

LLaVA-related items in transformers: ['LlavaConfig', 'LlavaForConditionalGeneration', 'LlavaImageProcessor', 'LlavaImageProcessorFast', 'LlavaModel', 'LlavaNextConfig', 'LlavaNextForConditionalGeneration', 'LlavaNextImageProcessor', 'LlavaNextImageProcessorFast', 'LlavaNextModel', 'LlavaNextPreTrainedModel', 'LlavaNextProcessor', 'LlavaNextVideoConfig', 'LlavaNextVideoForConditionalGeneration', 'LlavaNextVideoImageProcessor', 'LlavaNextVideoModel', 'LlavaNextVideoPreTrainedModel', 'LlavaNextVideoProcessor', 'LlavaNextVideoVideoProcessor', 'LlavaOnevisionConfig', 'LlavaOnevisionForConditionalGeneration', 'LlavaOnevisionImageProcessor', 'LlavaOnevisionImageProcessorFast', 'LlavaOnevisionModel', 'LlavaOnevisionPreTrainedModel', 'LlavaOnevisionProcessor', 'LlavaOnevisionVideoProcessor', 'LlavaPreTrainedModel', 'LlavaProcessor', 'VideoLlavaConfig', 'VideoLlavaForConditionalGeneration', 'VideoLlavaImageProcessor', 'VideoLlavaM

In [4]:
import subprocess
import sys

# Uninstall both system and user Pillow
print("Removing old Pillow installations...")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "pillow", "PIL"])

# Install latest Pillow to user directory
print("\nInstalling Pillow 10.x to user directory...")
subprocess.run([sys.executable, "-m", "pip", "install", "--user", "--force-reinstall", "pillow>=10.0.0"])

print("\n✅ Done! MUST restart kernel now!")
print("After restart, verify with: import PIL; print(PIL.__version__)")

Removing old Pillow installations...
Found existing installation: Pillow 9.0.1
Uninstalling Pillow-9.0.1:


ERROR: Exception:
Traceback (most recent call last):
  File "/usr/lib/python3.10/shutil.py", line 816, in move
    os.rename(src, real_dst)
PermissionError: [Errno 13] Permission denied: '/usr/lib/python3/dist-packages/PIL' -> '/tmp/pip-uninstall-msp2wjpu'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/ubuntu/.local/lib/python3.10/site-packages/pip/_internal/cli/base_command.py", line 107, in _run_wrapper
    status = _inner_run()
  File "/home/ubuntu/.local/lib/python3.10/site-packages/pip/_internal/cli/base_command.py", line 98, in _inner_run
    return self.run(options, args)
  File "/home/ubuntu/.local/lib/python3.10/site-packages/pip/_internal/commands/uninstall.py", line 105, in run
    uninstall_pathset = req.uninstall(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/pip/_internal/req/req_install.py", line 675, in uninstall
    uninstalled_pathset.remove(auto_confirm, verbose)
  File "/home/ubuntu/.


Installing Pillow 10.x to user directory...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 96.4 MB/s  0:00:00



✅ Done! MUST restart kernel now!
After restart, verify with: import PIL; print(PIL.__version__)


In [1]:

# Cell 2: Load LLaVA Model
# ===============================
#   LOAD LLAVA MODEL
# ===============================
import transformers
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "llava-hf/llava-1.5-7b-hf"

# Use the class objects directly from transformers module
LlavaProcessor = transformers.LlavaProcessor
LlavaForConditionalGeneration = transformers.LlavaForConditionalGeneration

processor = LlavaProcessor.from_pretrained(MODEL_NAME)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    output_attentions=True
)

print(f"✅ Loaded LLaVA successfully on {device}")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-01-08 05:15:19.022056: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767849319.075130    5016 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767849319.091306    5016 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767849319.188400    5016 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00

✅ Loaded LLaVA successfully on cuda


In [2]:

# Cell 3: Download VQA Dataset
# ===============================
#   DOWNLOAD VQA DATA
# ===============================
import os

os.makedirs("data/raw", exist_ok=True)

# Download annotations
os.system("wget -O data/raw/v2_mscoco_train2014_annotations.zip https://cvmlp.s3.amazonaws.com/vqa/mscoco/vqa/v2_Annotations_Train_mscoco.zip")

# Download questions
os.system("wget -O data/raw/v2_OpenEnded_mscoco_train2014_questions.zip https://cvmlp.s3.amazonaws.com/vqa/mscoco/vqa/v2_Questions_Train_mscoco.zip")

print("✅ Downloaded raw VQA dataset")


--2026-01-08 05:16:01--  https://cvmlp.s3.amazonaws.com/vqa/mscoco/vqa/v2_Annotations_Train_mscoco.zip
Resolving cvmlp.s3.amazonaws.com (cvmlp.s3.amazonaws.com)... 3.5.16.126, 16.15.223.178, 52.216.217.113, ...
Connecting to cvmlp.s3.amazonaws.com (cvmlp.s3.amazonaws.com)|3.5.16.126|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 21708861 (21M) [application/zip]
Saving to: ‘data/raw/v2_mscoco_train2014_annotations.zip’

     0K .......... .......... .......... .......... ..........  0% 26.2M 1s
    50K .......... .......... .......... .......... ..........  0% 14.4M 1s
   100K .......... .......... .......... .......... ..........  0% 15.1M 1s
   150K .......... .......... .......... .......... ..........  0% 27.7M 1s
   200K .......... .......... .......... .......... ..........  1% 26.6M 1s
   250K .......... .......... .......... .......... ..........  1% 15.5M 1s
   300K .......... .......... .......... .......... ..........  1% 26.9M 1s
   350K ..........

✅ Downloaded raw VQA dataset


. 99%  258K 0s
 21050K .......... .......... .......... .......... .......... 99%  108M 0s
 21100K .......... .......... .......... .......... .......... 99%  271M 0s
 21150K .......... .......... .......... .......... .......... 99% 52.9M 0s
 21200K                                                       100%  114G=0.6s

2026-01-08 05:16:02 (31.9 MB/s) - ‘data/raw/v2_mscoco_train2014_annotations.zip’ saved [21708861/21708861]

--2026-01-08 05:16:02--  https://cvmlp.s3.amazonaws.com/vqa/mscoco/vqa/v2_Questions_Train_mscoco.zip
Resolving cvmlp.s3.amazonaws.com (cvmlp.s3.amazonaws.com)... 16.182.67.81, 3.5.16.126, 52.216.217.113, ...
Connecting to cvmlp.s3.amazonaws.com (cvmlp.s3.amazonaws.com)|16.182.67.81|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7239401 (6.9M) [application/zip]
Saving to: ‘data/raw/v2_OpenEnded_mscoco_train2014_questions.zip’

     0K .......... .......... .......... .......... ..........  0% 41.8M 0s
    50K .......... .......... .......

In [3]:

# Cell 4: Extract Dataset
# ===============================
#   EXTRACT RAW VQA JSON FILES
# ===============================
import zipfile

for zip_file in [
    "data/raw/v2_mscoco_train2014_annotations.zip",
    "data/raw/v2_OpenEnded_mscoco_train2014_questions.zip"
]:
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall("data/raw")

print("✅ Extracted dataset to data/raw/")


✅ Extracted dataset to data/raw/


In [5]:

# Cell 5: Process & Filter Dataset
# ===============================
#   PROCESS VQA DATASET
# ===============================
import os
import json

annotations_path = "data/raw/v2_mscoco_train2014_annotations.json"
questions_path = "data/raw/v2_OpenEnded_mscoco_train2014_questions.json"
processed_path = "data/processed/filtered_vqa_with_links.json"

CATEGORY_MAPPING = {
    "what is this": "object identification",
    "what color": "color recognition",
    "how many": "counting",
    "what kind": "classification",
    "where": "location",
    "who": "person identification",
    "when": "time-related",
    "why": "reasoning",
    "which": "comparison",
    "other": "other",
}

def categorize_question(question_text, answer):
    for keyword, category in CATEGORY_MAPPING.items():
        if keyword in question_text.lower():
            return category
    return "yes/no" if answer.lower() in ["yes", "no"] else "other"

# ---- Load dataset ----
with open(annotations_path, "r") as f:
    annotations = json.load(f)
with open(questions_path, "r") as f:
    questions = json.load(f)

print(f"✅ Loaded {len(annotations['annotations'])} annotations and {len(questions['questions'])} questions")

question_map = {q["question_id"]: q["question"] for q in questions["questions"]}

# ---- Filter entries ----
filtered_data = []
for ann in annotations["annotations"][:5000]:  # first 5K = fast testing
    qid = ann["question_id"]
    img_id = str(ann["image_id"]).zfill(12)
    q_text = question_map.get(qid, "Unknown question")
    img_url = f"http://images.cocodataset.org/train2014/COCO_train2014_{img_id}.jpg"

    filtered_data.append({
        "image_id": img_id,
        "image_url": img_url,
        "question_id": qid,
        "question_text": q_text,
        "category": categorize_question(q_text, ann["multiple_choice_answer"]),
        "answer": ann["multiple_choice_answer"],
    })

os.makedirs(os.path.dirname(processed_path), exist_ok=True)
with open(processed_path, "w") as f:
    json.dump(filtered_data, f, indent=2)

print(f"✅ Processed {len(filtered_data)} entries and saved to {processed_path}")



✅ Loaded 443757 annotations and 443757 questions
✅ Processed 5000 entries and saved to data/processed/filtered_vqa_with_links.json


In [6]:

# Cell 6: Test the Model Quickly
# ===============================
#   TEST MODEL ON ONE EXAMPLE
# ===============================
from PIL import Image
import requests
import torch
import re
import json

with open("data/processed/filtered_vqa_with_links.json", "r") as f:
    vqa_data = json.load(f)

example = vqa_data[0]
img_url = example["image_url"]
question = example["question_text"]
answer = example["answer"]

print("🖼️ Image URL:", img_url)
print("❓ Question:", question)
print("✅ GT Answer:", answer)

image = Image.open(requests.get(img_url, stream=True).raw).convert("RGB").resize((224, 224))

def clean_pred(pred, ref=None):
    pred = pred.strip()
    if ref and ref.lower() in pred.lower():
        return ref.lower()
    pred = re.sub(r"[^a-z]", " ", pred).strip().split()
    return pred[-1].lower() if pred else ""

prompt = f"<image>\n{question}"
inputs = processor(text=prompt, images=[image], return_tensors="pt").to(device)

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=20)
pred = processor.decode(output_ids[0], skip_special_tokens=True)
cleaned = clean_pred(pred, answer)
correct = int(cleaned == answer.lower())

print("\n🧠 Raw Prediction:", pred)
print("✂️ Cleaned:", cleaned)
print("✅ Correct?", bool(correct))



🖼️ Image URL: http://images.cocodataset.org/train2014/COCO_train2014_000000458752.jpg
❓ Question: What is this photo taken looking through?
✅ GT Answer: net

🧠 Raw Prediction: 
What is this photo taken looking through?

A mesh net
✂️ Cleaned: net
✅ Correct? True


In [7]:

# Cell 7: Extract Attention + Spatial Features
# ===============================
#   IDEA 4 — EXTRACT ATTENTION + SPATIAL FEATURES
# ===============================
import torch
import numpy as np
import re
import json
import os
import requests
from tqdm import tqdm
from PIL import Image

NUM_SAMPLES = 3000  # increase for better classifier accuracy
LAYER_WINDOW = 3
IMAGE_SIZE = (224, 224)
device = "cuda" if torch.cuda.is_available() else "cpu"

def clean_answer(pred, ref):
    pred, ref = pred.strip().lower(), ref.strip().lower()
    if ref in pred:
        return ref
    pred = re.sub(r"[^a-z]", " ", pred).strip().split()
    return pred[-1] if pred else ""

with open("data/processed/filtered_vqa_with_links.json", "r") as f:
    vqa_data = json.load(f)

features = []
for i, entry in enumerate(tqdm(vqa_data[:NUM_SAMPLES], desc="Extracting attention + spatial features")):
    try:
        image = Image.open(requests.get(entry["image_url"], stream=True).raw).convert("RGB").resize(IMAGE_SIZE)
        prompt = f"<image>\n{entry['question_text']}"
        inputs = processor(text=prompt, images=[image], return_tensors="pt").to(device)

        with torch.no_grad():
            out = model(**inputs, output_attentions=True)
            gen = model.generate(**inputs, max_new_tokens=20)

        pred = processor.decode(gen[0], skip_special_tokens=True)
        cleaned = clean_answer(pred, entry["answer"])
        correct = int(cleaned == entry["answer"].lower())

        attns = out.attentions
        mean_attention = mean_entropy = entropy_var = None
        if attns:
            last_layers = attns[-LAYER_WINDOW:]
            mean_attention = float(torch.stack([a.mean() for a in last_layers]).mean())
            entropies = []
            for layer in last_layers:
                probs = layer.softmax(dim=-1)
                ent = -torch.sum(probs * torch.log(probs + 1e-12), dim=-1).mean()
                entropies.append(ent.item())
            mean_entropy = float(np.mean(entropies))
            entropy_var = float(np.var(entropies))

        # Spatial stats
        try:
            att = attns[-1].mean(dim=1)[0]
            img_tokens = att[1:577, 1:577]
            grid = int(np.sqrt(img_tokens.shape[0]))
            att_grid = img_tokens.mean(dim=0).view(grid, grid).cpu().numpy()
            att_grid = np.maximum(att_grid, 0)
            att_grid /= att_grid.sum() + 1e-12
            concentration = float(np.sum(att_grid**2))
            rows, cols = np.indices(att_grid.shape)
            com_r = (rows * att_grid).sum() / att_grid.sum()
            com_c = (cols * att_grid).sum() / att_grid.sum()
            center_dist = float(np.sqrt((com_r - grid/2)**2 + (com_c - grid/2)**2))
            flat = att_grid.flatten()
            spatial_entropy = float(-np.sum(flat * np.log(flat + 1e-12)))
        except Exception:
            concentration = center_dist = spatial_entropy = None

        features.append({
            "question_id": entry["question_id"],
            "mean_strength": mean_attention,
            "mean_entropy": mean_entropy,
            "entropy_var": entropy_var,
            "concentration": concentration,
            "center_dist": center_dist,
            "spatial_entropy": spatial_entropy,
            "correct": correct
        })

        del inputs, out, gen, image
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"⚠️ Skipped {i}: {e}")

os.makedirs("idea_4/results", exist_ok=True)
with open("idea_4/results/attention_spatial_features.json", "w") as f:
    json.dump(features, f, indent=2)

print(f"✅ Saved {len(features)} attention + spatial feature records")



Extracting attention + spatial features: 100%|██████████| 3000/3000 [28:45<00:00,  1.74it/s]

✅ Saved 3000 attention + spatial feature records


In [8]:

# Cell 8: Train Improved Classifier
# ============================================
# IDEA 4 — CROSS-ATTENTION + SPATIAL CLASSIFIER
# ============================================
import os
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from xgboost import XGBClassifier

RESULTS_DIR = "idea_4/results"
feat_path = os.path.join(RESULTS_DIR, "attention_spatial_features.json")

# --- Load data ---
with open(feat_path, "r") as f:
    df = pd.DataFrame(json.load(f))
df = df.dropna(subset=["mean_strength", "mean_entropy"])
df = df[df["mean_strength"] > 0]
print("✅ Loaded", len(df), "valid samples")
print("Label balance:", df["correct"].value_counts().to_dict())

# --- Feature engineering ---
df["entropy_ratio"] = df["mean_entropy"] / (df["mean_strength"] + 1e-9)
df["focus_score"] = df["mean_strength"] / (df["entropy_var"] + 1e-9)
df["entropy_log"] = np.log1p(df["mean_entropy"])
df["strength_log"] = np.log1p(df["mean_strength"])
df["entropy_strength_interaction"] = df["mean_entropy"] * df["mean_strength"]

features = [
    "mean_strength", "mean_entropy", "entropy_var",
    "concentration", "center_dist", "spatial_entropy",
    "entropy_ratio", "focus_score",
    "entropy_log", "strength_log", "entropy_strength_interaction"
]
X = df[features].fillna(0)
y = df["correct"].astype(int)

scaler = StandardScaler()
poly = PolynomialFeatures(degree=2, include_bias=False)
X_scaled = scaler.fit_transform(X)
X_poly = poly.fit_transform(X_scaled)

X_train, X_test, y_train, y_test = train_test_split(
    X_poly, y, test_size=0.2, stratify=y, random_state=42
)

xgb = XGBClassifier(
    n_estimators=600, max_depth=10, learning_rate=0.03,
    subsample=0.9, colsample_bytree=0.9,
    reg_lambda=1.0, reg_alpha=0.3,
    random_state=42, n_jobs=-1, verbosity=0
)
rf = RandomForestClassifier(
    n_estimators=400, max_depth=16,
    min_samples_split=3, min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=42, n_jobs=-1
)
gb = GradientBoostingClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42
)

clf = StackingClassifier(
    estimators=[('xgb', xgb), ('rf', rf)],
    final_estimator=gb, n_jobs=-1
)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\n🎯 Accuracy: {acc:.3f}")
print(classification_report(y_test, y_pred, digits=3))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(clf, X_poly, y, cv=cv, scoring="accuracy", n_jobs=-1)
print(f"🔁 Cross-validated Accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

joblib.dump(clf, os.path.join(RESULTS_DIR, "spatial_confidence_classifier.pkl"))
joblib.dump(scaler, os.path.join(RESULTS_DIR, "spatial_confidence_scaler.pkl"))
joblib.dump(poly, os.path.join(RESULTS_DIR, "spatial_confidence_poly.pkl"))
print("✅ Saved optimized spatial classifier.")

✅ Loaded 3000 valid samples
Label balance: {1: 1689, 0: 1311}

🎯 Accuracy: 0.555
              precision    recall  f1-score   support

           0      0.488     0.393     0.436       262
           1      0.591     0.680     0.633       338

    accuracy                          0.555       600
   macro avg      0.540     0.537     0.534       600
weighted avg      0.546     0.555     0.547       600

🔁 Cross-validated Accuracy: 0.514 ± 0.015
✅ Saved optimized spatial classifier.
